# 01 — 데이터 전처리 (Preprocessing)

> 원본 데이터를 정제하고, 모델 학습을 위한 입력 데이터셋을 생성합니다.

## 구성

1) 데이터 로드 (rawdata.csv)  
2) 결측치 및 이상값 처리  
3) 파생 변수 생성 (month_sin, month_cos 등)  
4) 시나리오 그룹 생성 (scenario_group)  
5) 스케일링 (RobustScaler)  
6) EDA 요약 출력  

## 산출물

- data/processed/processed.csv  
  → 스케일된 특징 + 타깃 변수 + scenario_group  

- model/robust_scaler.joblib  
  → 학습에 사용된 스케일러  

(참고)  
- data/raw/rawdata.csv (원본 데이터)

In [1]:
# =========================
# 0) Imports & Settings
# =========================
import warnings, os, random
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42); random.seed(42)

# 폴더 생성
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("model", exist_ok=True)

print("환경 준비 완료")


환경 준비 완료


In [4]:
# =========================
# 1) Load & Basic Prep
# =========================
df = pd.read_csv("rawdata.csv")
targets = ["Leaf_TPC","Root_TPC","Leaf_TFC","Root_TFC"]

def ensure_month_feats(df, month_col="month"):
    if "month_sin" not in df.columns or "month_cos" not in df.columns:
        if month_col in df.columns:
            m = df[month_col].astype(float)
            df["month_sin"] = np.sin(2*np.pi*m/12)
            df["month_cos"] = np.cos(2*np.pi*m/12)
    return df

df = ensure_month_feats(df)

In [5]:
# =========================
# 2) 시나리오 그룹 라벨
# =========================
def scenario_label(co2):
    if co2 < 633:     # ~432와 834의 중간
        return "SSP1-2.6"
    elif co2 < 961:   # ~834와 1089의 중간
        return "SSP3-7.0"
    else:
        return "SSP5-8.5"

if "scenario_group" not in df.columns and "CO2ppm" in df.columns:
    df["scenario_group"] = df["CO2ppm"].apply(scenario_label)


In [6]:
# =========================
# 3) Lightweight EDA Summary
# =========================
print("Data shape:", df.shape)
print("\nMissing counts (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nTargets describe():")
print(df[targets].describe())

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_top = df[num_cols].corr().loc[targets]
print("\nSimple corr rows for targets:")
print(corr_top.iloc[:, :8])

Data shape: (405, 28)

Missing counts (top 10):
month                   0
CO2ppm                  0
month_cos               0
month_sin               0
Root_TFC                0
Leaf_TFC                0
Root_TPC                0
Leaf_TPC                0
Root_ExtractionYield    0
Leaf_ExtractionYield    0
dtype: int64

Targets describe():
         Leaf_TPC    Root_TPC    Leaf_TFC    Root_TFC
count  405.000000  405.000000  405.000000  405.000000
mean     7.708358    5.147294    4.670188    0.605360
std      0.506566    0.549619    2.575212    0.136367
min      6.247000    4.347000    1.199000    0.391000
25%      7.369000    4.678000    2.129000    0.492000
50%      7.814000    5.104000    5.242000    0.583000
75%      8.027000    5.293000    7.142000    0.694000
max      8.526000    6.396000    8.007000    0.861000

Simple corr rows for targets:
             month    CO2ppm      Temp     Humid       VPD     Chl_a  \
Leaf_TPC -0.377048  0.268194  0.231998  0.052587  0.232120 -0.353476 

In [ ]:
# =========================
# 4) Scaling
# =========================
final_features = [
    "Temp","Humid","CO2ppm",
    "TChl","Car",
    "Dio-RC","Eto-RC",
    "Fv-Fm",
    "Leaf_ExtractionYield","Root_ExtractionYield",
    "month_sin","month_cos"
]

month_feats = ["month_sin","month_cos"]
num_feats = [f for f in final_features if f not in month_feats]

scaler = RobustScaler()
X_scaled = df.copy()
X_scaled[num_feats] = scaler.fit_transform(df[num_feats])

X = X_scaled[final_features]
y = df[targets]

print("\nX shape:", X.shape)
print("y shape:", y.shape)


X shape: (405, 12)
y shape: (405, 4)


In [8]:
# =========================
# 5) Save Artifacts
# =========================
from joblib import dump

df_out = pd.concat([
    X_scaled[final_features],  # 스케일된 피처
    df[targets],               # 원본 타깃
    df[["scenario_group"]]     # 메타(그룹)
], axis=1)

df_out.to_csv("data/processed/processed.csv", index=False, encoding="utf-8-sig",
              float_format="%.6f", na_rep="NA")
dump(scaler, "model/robust_scaler.joblib")

print("\nSaved → data/processed/processed.csv")
print("Saved → model/robust_scaler.joblib")
print("Final shape:", df_out.shape)
df_out


Saved → data/processed/processed.csv
Saved → model/robust_scaler.joblib
Final shape: (405, 17)


,Temp,Humid,CO2ppm,TChl,Car,Dio-RC,Eto-RC,Fv-Fm,Leaf_ExtractionYield,Root_ExtractionYield,month_sin,month_cos,Leaf_TPC,Root_TPC,Leaf_TFC,Root_TFC,scenario_group
0,-2.496005,4.250354,-0.781807,1.419929,1.346154,-0.476636,-0.652406,0.750000,0.431579,2.296296,0.5,-8.660254e-01,7.476,6.270,5.217,0.861,SSP1-2.6
1,-2.494877,4.242837,-0.792098,1.615658,1.500000,-0.336449,-0.406417,0.583333,0.547368,2.555556,0.5,-8.660254e-01,7.369,6.396,5.257,0.836,SSP1-2.6
2,-2.492242,4.108514,-0.795823,1.811388,1.525641,-0.406542,-0.208556,0.666667,0.610526,2.851852,0.5,-8.660254e-01,7.369,6.396,5.242,0.841,SSP1-2.6
3,-2.495075,4.018841,-0.755010,1.701068,1.551282,-0.710280,-0.775401,1.125000,0.431579,2.296296,0.5,-8.660254e-01,7.476,6.270,5.217,0.861,SSP1-2.6
4,-2.495136,4.417593,-0.782264,2.170819,1.858974,-0.448598,-0.604278,0.833333,0.547368,2.555556,0.5,-8.660254e-01,7.369,6.396,5.257,0.836,SSP1-2.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,0.413692,-0.811467,0.397032,-1.373665,-1.089744,2.420561,-0.711230,-3.750000,-0.821053,0.703704,-1.0,-1.836970e-16,7.743,5.277,1.455,0.513,SSP5-8.5
401,0.412073,-0.797786,0.401948,-0.409253,-0.500000,4.500000,-0.930481,-9.333333,-0.826316,0.685185,-1.0,-1.836970e-16,7.760,5.245,1.450,0.507,SSP5-8.5
402,0.420819,-1.046918,0.393541,-1.569395,-0.692308,4.140187,-1.010695,-7.041667,-0.831579,0.666667,-1.0,-1.836970e-16,7.814,5.324,1.460,0.518,SSP5-8.5
403,0.410129,-1.858750,0.420847,-1.419929,-0.961538,1.985981,-0.481283,-3.541667,-0.821053,0.703704,-1.0,-1.836970e-16,7.743,5.277,1.455,0.513,SSP5-8.5
